## 1. 核心思想
- 建立**两级索引结构**：第一层是文档/章节的**摘要索引**，第二层是详细的**文档块索引**。检索时先通过摘要层快速定位相关文档，再在目标文档内精确检索具体块。类似于"先查目录，再翻具体页"。
- 它的目标在于**“先粗后细”**。如果不做分层，直接在一千万个切片中检索，噪音非常大（比如不同文档中都有“第一章”）。通过先定位到“文档级”，我们可以构建一个“隔音墙”，确保第二步的检索只在相关的文档内部进行。

## 2. 核心逻辑伪代码

In [ ]:
def hierarchical_search(user_query):
    """
    分层检索核心逻辑：
    1. 摘要层检索 -> 锁定文档 ID
    2. 内容层检索 -> 仅在锁定的文档 ID 范围内搜索
    """

    # Step 1: 第一层检索 (摘要索引)
    # 目的：找到"哪篇文档"在讲这件事
    summary_vector = embed(user_query)
    relevant_docs = summary_index.search(summary_vector, top_k=3)

    # 提取文档 ID 列表，例如 ["doc_01", "doc_05"]
    target_doc_ids = [doc.id for doc in relevant_docs]

    # Step 2: 第二层检索 (详细块索引)
    # 关键点：使用 filter 表达式进行"前置过滤"
    # 语义：Search query WHERE doc_id IN target_doc_ids
    chunk_vector = embed(user_query)

    final_chunks = chunk_index.search(
        chunk_vector,
        top_k=5,
        filter=f"doc_id in {target_doc_ids}"  # 核心：圈定范围
    )

    return final_chunks

## 3. 完整实现（Milvus + OpenAI）
我们需要在 Milvus 中创建两个集合 (Collections)：

rag_summaries：存文档摘要向量，Metadata 包含 doc_id。

rag_chunks：存具体切片向量，Metadata 包含 doc_id（作为外键）。

### 3.1 配置

In [1]:
import os
import uuid
from pymilvus import MilvusClient, DataType
from openai import OpenAI
import numpy as np

# --- 配置部分 ---
os.environ["OPENAI_API_KEY"] = "sk-OdqRypFlfJLKYvLmV6GsG9j0u6CRFBYKErn4xV1Wm0R3q0y9"  # 替换你的 Key
os.environ["OPENAI_BASE_URL"] = "https://api.openai-proxy.org/v1"
client = OpenAI()
milvus_client = MilvusClient(uri="http://111.228.53.183:19530")

SUMMARY_COLLECTION = "rag_summaries" # 第一层索引
CHUNK_COLLECTION = "rag_chunks"      # 第二层索引
DIMENSION = 3072

### 3.2 核心函数封装

In [3]:
def get_embedding(text):
    response = client.embeddings.create(input=text, model="text-embedding-3-large")
    return response.data[0].embedding

def generate_summary(text):
    """使用 LLM 生成文档摘要"""
    prompt = f"请为以下文档生成一段简短的摘要（50字以内），概括其核心主题和适用场景：\n\n{text[:1000]}"
    res = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}]
    )
    return res.choices[0].message.content

def split_text(text, chunk_size=100):
    """简单的切分函数"""
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

### 3.3 初始化两个集合

In [4]:
def init_collections():
    # 1. 摘要集合 (Level 1)
    if milvus_client.has_collection(SUMMARY_COLLECTION):
        milvus_client.drop_collection(SUMMARY_COLLECTION)
    milvus_client.create_collection(
        collection_name=SUMMARY_COLLECTION,
        dimension=DIMENSION,
        auto_id=True,
        enable_dynamic_field=True # 允许存 doc_id
    )

    # 2. 切片集合 (Level 2)
    if milvus_client.has_collection(CHUNK_COLLECTION):
        milvus_client.drop_collection(CHUNK_COLLECTION)
    milvus_client.create_collection(
        collection_name=CHUNK_COLLECTION,
        dimension=DIMENSION,
        auto_id=True,
        enable_dynamic_field=True # 允许存 doc_id 用于过滤
    )
    print("索引结构初始化完成 (Summary Layer + Chunk Layer)。\n")

init_collections()

索引结构初始化完成 (Summary Layer + Chunk Layer)。



### 3.4 构建索引

In [5]:
# --- Step 2: 索引构建 (Ingestion) ---

# 模拟两篇完全不同领域的文档
documents = [
    {
        "title": "员工手册 - 休假制度",
        "content": "本公司员工享有年假、病假和事假。入职满一年者享有 5 天带薪年假。病假需提供三甲医院证明，且病假期间工资按 80% 发放。事假需提前 3 天向部门主管申请，事假期间无工资。法定节假日按国家规定执行。"
    },
    {
        "title": "API 开发指南 - 认证模块",
        "content": "我们的 API 使用 OAuth 2.0 协议进行认证。开发者需要在 Header 中携带 'Authorization: Bearer <token>'。Token 有效期为 2 小时，过期后需通过 Refresh Token 获取新令牌。如果返回 401 错误，请检查 Token 是否过期或签名是否正确。"
    }
]



print("正在处理文档并构建分层索引...")

for doc in documents:
    # 1. 生成唯一文档 ID
    doc_id = str(uuid.uuid4())
    content = doc["content"]
    title = doc["title"]

    # --- 第一层：摘要索引 ---
    summary = generate_summary(content)
    summary_vec = get_embedding(summary)

    print(f" -> [Level 1] 文档 '{title}' 生成的摘要: {summary}")
    milvus_client.insert(
        collection_name=SUMMARY_COLLECTION,
        data=[{
            "vector": summary_vec,
            "doc_id": doc_id,
            "title": title,
            "summary_text": summary
        }]
    )

    # --- 第二层：详细块索引 ---
    chunks = split_text(content, chunk_size=50)
    chunk_data = []
    for chunk in chunks:
        chunk_data.append({
            "vector": get_embedding(chunk),
            "doc_id": doc_id,  # 关键外键：关联回第一层
            "chunk_text": chunk,
            "title": title
        })

    milvus_client.insert(collection_name=CHUNK_COLLECTION, data=chunk_data)
    print(f"    [Level 2] 插入了 {len(chunks)} 个详细切片。\n")

print("-" * 50)

正在处理文档并构建分层索引...
 -> [Level 1] 文档 '员工手册 - 休假制度' 摘要生成的摘要: 这份文档主要介绍了公司的休假政策，包括年假、病假和事假的具体规定，以及对应的带薪待遇和申请流程，适用于公司员工了解和参照。
    [Level 2] 插入了 2 个详细切片。

 -> [Level 1] 文档 'API 开发指南 - 认证模块' 摘要生成的摘要: 该文档介绍了使用OAuth 2.0协议进行认证的API，包括Token的携带方式、有效期、刷新方法以及错误处理。主要适用于API开发者进行接口认证和问题排查。
    [Level 2] 插入了 4 个详细切片。

--------------------------------------------------


## 3.5. 检索
- 定义检索函数

In [7]:
# --- Step 3: 两级检索逻辑 (Retrieval) ---

def hierarchical_rag(query):
    print(f"用户查询: {query}")
    query_vec = get_embedding(query)

    # === Level 1: 摘要检索 (定位文档) ===
    # 我们先找 Top 1 最相关的文档 (实际场景可以是 Top 3 or 5)
    summary_res = milvus_client.search(
        collection_name=SUMMARY_COLLECTION,
        data=[query_vec],
        limit=1,
        output_fields=["doc_id", "title", "summary_text"]
    )

    if not summary_res:
        return "未找到相关文档。"

    top_doc = summary_res[0][0]["entity"]
    target_doc_id = top_doc["doc_id"]
    target_doc_title = top_doc["title"]

    print(f" -> [Level 1 命中] 锁定文档: 《{target_doc_title}》")
    print(f"    (原因: 摘要与查询匹配度高)")

    # === Level 2: 切片检索 (定位细节) ===
    # 核心：使用 expr (Expression) 进行过滤
    # 只有 doc_id 等于刚才找到的那个 ID 的切片，才有资格被检索
    filter_expr = f'doc_id == "{target_doc_id}"'

    chunk_res = milvus_client.search(
        collection_name=CHUNK_COLLECTION,
        data=[query_vec],
        limit=2, # 在文档内找最相关的 2 句话
        filter=filter_expr, # <--- 核心过滤器
        output_fields=["chunk_text"]
    )

    retrieved_contexts = [hit["entity"]["chunk_text"] for hit in chunk_res[0]]

    # === Level 3: LLM 生成 ===
    final_prompt = f"""
    基于以下文档片段回答问题。

    文档标题: {target_doc_title}
    内容片段: {'...'.join(retrieved_contexts)}

    问题: {query}
    """

    print(" -> [Level 3 生成] 正在请求 LLM...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": final_prompt}]
    )

    return response.choices[0].message.content

## 3.6 检索
- 测试

In [10]:
# 场景 1：查询 IT 技术问题
# 预期：
# 1. 摘要层忽略“休假制度”，命中“API 开发指南”
# 2. 切片层在 API 文档中找到“401错误”相关切片
print("\n=== 测试 1：技术类查询 ===")
ans1 = hierarchical_rag("接口报错 401 是什么原因？")
print(f"\n>> RAG 回答:\n{ans1}")

# 场景 2：查询 HR 问题
# 预期：
# 1. 摘要层命中“员工手册”
# 2. 切片层在手册中找到“病假工资”相关切片
print("\n\n=== 测试 2：行政类查询 ===")
ans2 = hierarchical_rag("生病请假扣工资吗？")
print(f"\n>> RAG 回答:\n{ans2}")


=== 测试 1：技术类查询 ===
用户查询: 接口报错 401 是什么原因？
 -> [Level 1 命中] 锁定文档: 《API 开发指南 - 认证模块》
    (原因: 摘要与查询匹配度高)
 -> [Level 3 生成] 正在请求 LLM...

>> RAG 回答:
接口报错 401 的原因可能是 Token 已经过期或签名无效。请检查您使用的 Token 是否仍在有效期内，若已过期，则需要通过 Refresh Token 获取新的令牌。


=== 测试 2：行政类查询 ===
用户查询: 生病请假扣工资吗？
 -> [Level 1 命中] 锁定文档: 《员工手册 - 休假制度》
    (原因: 摘要与查询匹配度高)
 -> [Level 3 生成] 正在请求 LLM...

>> RAG 回答:
根据文档片段，病假期间需要提供三甲医院的证明，但没有明确说明病假期间是否扣工资。因此，文档中未提及病假期间工资的具体发放情况。建议向人事部门确认相关政策。
